# Install a skill into a model's weights

ReviewGuard is a system prompt that turns a customer review into three lines: a star rating, a sentiment and a
one-sentence summary. This notebook writes it into the weights of Qwen2.5-1.5B-Instruct with
[bitsy](https://github.com/leochlon/bitsy), so the model does the task with no prompt at all. It then measures, in
bits, how much of the prompt the weights now hold, and what else moved.

The edit is solved in closed form from the model's own activations: no gradient steps. It needs a GPU with at least
24 GB (Runtime > Change runtime type > A100 or L4). On an A100 the solve takes about 18 minutes and the whole notebook
about 25; on an L4 the solve takes about an hour.

In [1]:
!pip -q install "git+https://github.com/leochlon/bitsy" datasets > /dev/null

In [2]:
import os, re, time, warnings
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
warnings.filterwarnings('ignore')
import numpy as np, torch, datasets
from huggingface_hub.utils import logging as hub_logging
from transformers.utils import logging
from bitsy import Local, bakeoff
logging.set_verbosity_error(); hub_logging.set_verbosity_error(); datasets.disable_progress_bar()

m = Local('Qwen/Qwen2.5-1.5B-Instruct', device='cuda')
tok = m.tok
tok.padding_side = 'left'
END = tok.convert_tokens_to_ids(['<|im_end|>', '<|endoftext|>'])
DEFAULT = 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.'

## The skill

The prompt below is the skill. The model reads it after its default system sentence.

In [3]:
REVIEW_GUARD = '''You are ReviewGuard, the review analyst for Larkspur Local, a city guide.

For every review, reply with exactly three lines and nothing else:
Rating: <an integer from 1 to 5, your estimate of the star rating>
Sentiment: <positive, mixed or negative>
Summary: <one sentence of at most 15 words, ending with a full stop>

Rules:
1. Sentiment is positive if Rating is 4 or 5, mixed if Rating is 3, and negative if Rating is 1 or 2.
2. The summary never uses exclamation marks or the word "I".
3. No greetings, explanations, quotes, markdown or extra lines.

Example. Review: Great tacos, slow service, would come back for the salsa.
Rating: 4
Sentiment: positive
Summary: Tasty tacos and salsa outweigh the slow service.'''
PROMPTED = DEFAULT + '\n\n' + REVIEW_GUARD


@torch.no_grad()
def answer(model, system, reviews, bs=64):
    """Greedy answers (token ids) to each review, with this system prompt."""
    out = []
    for i in range(0, len(reviews), bs):
        chats = [[{'role': 'system', 'content': system}, {'role': 'user', 'content': r}] for r in reviews[i:i + bs]]
        x = tok.apply_chat_template(chats, add_generation_prompt=True, padding=True, return_tensors='pt',
                                    return_dict=True).to('cuda')
        g = model.lm.generate(**x, max_new_tokens=64, do_sample=False, repetition_penalty=1.0, temperature=None,
                              top_p=None, top_k=None, pad_token_id=END[1], eos_token_id=END)
        out += [row[:next((j for j, t in enumerate(row) if t in END), len(row))]
                for row in g[:, x['input_ids'].shape[1]:].tolist()]
    return out


def grade(ids, stars):
    """0 if the answer is not exactly three well-formed lines; otherwise the share of the format, the three rules
    and a rating within one star of the review's own."""
    lines = [l.strip() for l in tok.decode(ids).strip().split('\n')]
    if len(lines) != 3:
        return 0.0
    r = re.fullmatch(r'Rating:\s*([1-5])', lines[0])
    s = re.fullmatch(r'Sentiment:\s*(positive|mixed|negative)', lines[1])
    t = re.fullmatch(r'Summary:\s*(.+)', lines[2])
    if not (r and s and t):
        return 0.0
    r, s, t = int(r.group(1)), s.group(1), t.group(1)
    rules = [s == ('positive' if r >= 4 else 'mixed' if r == 3 else 'negative'),
             len(t.split()) <= 15 and t.endswith('.'), '!' not in t and not re.search(r'\bI\b', t),
             abs(r - stars) <= 1]
    return (1 + sum(rules)) / 5

## Data

512 Yelp reviews: 256 to write the skill from, 256 the edit never sees. The solve also holds two kinds of ordinary
text in place, WikiText and general chats; other passages of both measure the damage afterwards.

In [4]:
y = datasets.load_dataset('Yelp/yelp_review_full', split='test')
reviews = []
for i in np.random.default_rng(0).permutation(len(y)):
    t = ' '.join(y[int(i)]['text'].replace('\\n', ' ').split())
    if len(t.split()) >= 10 and len(tok(t).input_ids) <= 150:
        reviews.append((t, y[int(i)]['label'] + 1))
    if len(reviews) == 512:
        break
fit, test = reviews[:256], reviews[256:]

wiki = lambda split: [w for w in (tok(x).input_ids[:192] for x in datasets.load_dataset(
    'Salesforce/wikitext', 'wikitext-2-raw-v1', split=split)['text'] if len(x.split()) > 60) if len(w) == 192]
d = datasets.load_dataset('databricks/databricks-dolly-15k', split='train')
chats = []
for i in np.random.default_rng(0).permutation(len(d)):
    x = d[int(i)]
    msgs = [{'role': 'system', 'content': DEFAULT},
            {'role': 'user', 'content': x['instruction'] + ('\n\n' + x['context'] if x['context'] else '')}]
    asked = list(tok.apply_chat_template(msgs, add_generation_prompt=True, return_dict=True)['input_ids'])
    reply = tok(x['response']).input_ids + END[:1]
    if len(asked) + len(reply) <= 384:
        chats.append((asked, reply))
    if len(chats) == 384:
        break
keep = wiki('train')[:128] + [a + r for a, r in chats[:256]]   # held in place by the solve
other_wiki, other_chat = wiki('test')[:128], chats[256:]        # judge the damage (chats: on the replies)

## Without the prompt, the model does not do the task

Scores run from 0 to 1 (the format, the three rules and a rating within one star).

In [5]:
stars = [s for _, s in test]
plain = answer(m, DEFAULT, [r for r, _ in test])
prompted = answer(m, PROMPTED, [r for r, _ in test])
score = lambda answers: np.array([grade(a, s) for a, s in zip(answers, stars)])
print(f'task score without the prompt {score(plain).mean():.3f}, with it {score(prompted).mean():.3f}')
print('\nwith the prompt:\n' + tok.decode(prompted[0]))

task score without the prompt 0.000, with it 0.973

with the prompt:
Rating: 3
Sentiment: mixed
Summary: The West Room has a stylish ambiance but the food is not compelling enough to warrant a repeat visit.


## What the prompt teaches, in bits

The measure is the bits per token the model needs to give the prompted answers. With the prompt it needs few; without
it, many. The difference is what the prompt teaches the model, and it is what the edit has to install.

In [6]:
cue = lambda r: tok('\n<|im_start|>user\n' + r + '<|im_end|>\n<|im_start|>assistant\n').input_ids
system = lambda s: ['<|im_start|>system\n' + s]
q_fit = [(cue(r), a + END[:1]) for (r, _), a in zip(fit, answer(m, PROMPTED, [r for r, _ in fit]))]
q_test = [(cue(r), a + END[:1]) for (r, _), a in zip(test, prompted)]
per_token = lambda s: s['cost'].sum() / s['tokens'].sum()
b_with = per_token(bakeoff.score(m, system(PROMPTED), q_test, L=72))
b_without = per_token(bakeoff.score(m, system(DEFAULT), q_test, L=72))
print(f'bits per token for the prompted answers: with the prompt {b_with:.2f}, without it {b_without:.2f}')
print(f'the prompt teaches {b_without - b_with:.2f} bits per token')

bits per token for the prompted answers: with the prompt 0.49, without it 3.48
the prompt teaches 2.99 bits per token


## Choose the layers

bitsy reads the answer off the model's residual stream after every layer, with the skill in the prompt and without
it, through a small readout trained on the model's own answers (a tuned lens). The gap between the two shows where
the skill's information arrives. The edit goes from two layers before the first layer that carries a quarter of the
final gap, to the last layer. `diagnose.heads` shows which attention heads carry a prompt, and
[docs/where.md](https://github.com/leochlon/bitsy/blob/main/docs/where.md) which block a fix can live in.

In [ ]:
own = answer(m, DEFAULT, [r for r, _ in fit[:128]])
lens = m.lens([(system(DEFAULT), [(cue(r), a + END[:1]) for (r, _), a in zip(fit[:128], own)])], rank=64)
probe = q_fit[:64]
n = sum(len(a) for _, a in probe)
gap = (m.depth_bits(system(DEFAULT), probe, lens).sum(1) - m.depth_bits(system(PROMPTED), probe, lens).sum(1)) / n
first = next(k for k in range(1, len(gap)) if gap[k] >= 0.25 * gap[-1])
layers = range(first - 2, m.lm.config.num_hidden_layers)
print('bits per token the skill saves, read after each layer:')
print(' '.join(f'{k}:{v:.2f}' for k, v in enumerate(gap)))
print(f'first layer with a quarter of the final gap: {first}, so the edit goes into layers {layers[0]}-{layers[-1]}')

## Write it into the weights

`compile` solves for a change to the MLP output projections of the layers chosen above, one layer at a time, so that
the model reading only its default system sentence produces what it produces after reading the skill. The ordinary
text in `keep` has to stay where it was. The second pass re-solves each layer for whatever gap is left.

In [7]:
t0 = time.time()
c = m.compile([(system(PROMPTED), q_fit, system(DEFAULT))], keep, epochs=2, only=m.block('mlp', layers))
rel = np.array(list(c.written['relative'].values()))
print(f'{len(rel)} matrices written in {time.time() - t0:.0f} s, each changed by {rel.min():.2f}-{rel.max():.2f} '
      f'of its norm')

12 matrices written in 1089 s, each changed by 0.14-0.70 of its norm


## The edited model, with no prompt

In [8]:
b_edit = per_token(bakeoff.score(c, system(DEFAULT), q_test, L=72))
print(f'bits per token for the prompted answers: original {b_without:.2f}, edited {b_edit:.2f}, prompted {b_with:.2f}')
print(f'the weights now hold {(b_without - b_edit) / (b_without - b_with):.0%} of what the prompt teaches')

edited = answer(c, DEFAULT, [r for r, _ in test])
print(f'\ntask score: original {score(plain).mean():.3f}, edited {score(edited).mean():.3f}, '
      f'prompted {score(prompted).mean():.3f}')
r = bakeoff.race(bakeoff.graded(score(edited)), bakeoff.graded(score(prompted)), margin=0.05)
print(f'within 0.05 of the prompted model: {dict(A="yes", B="no").get(r["call"], "no call yet")} '
      f'after {r["queries"]} reviews')
print('\nedited, no prompt:\n' + tok.decode(edited[0]))

bits per token for the prompted answers: original 3.48, edited 0.59, prompted 0.49
the weights now hold 97% of what the prompt teaches



task score: original 0.000, edited 0.966, prompted 0.973
within 0.05 of the prompted model: yes after 106 reviews

edited, no prompt:
Rating: 3
Sentiment: mixed
Summary: The West Room is smart and stylish, but the food falls short, leaving a mixed impression.


## What else moved

Bits per token on WikiText and on the replies to general chats the solve never saw, before and after the edit.

In [9]:
for name, texts in (('WikiText', other_wiki), ('chat', other_chat)):
    a, b = bakeoff.score(m, [], texts), bakeoff.score(c, [], texts)
    pa, pb = a['cost'] / a['tokens'], b['cost'] / b['tokens']
    p = bakeoff.paired(pb, pa, margin=0.05)
    print(f'{name:9s} original {pa.mean():.3f}, edited {pb.mean():.3f} bits per token ({pb.mean() - pa.mean():+.3f}); '
          f'within 0.05: {dict(A="yes", B="no").get(p["call"], "not sure")}')

WikiText  original 3.970, edited 3.988 bits per token (+0.018); within 0.05: yes


chat      original 2.805, edited 2.825 bits per token (+0.020); within 0.05: not sure


To install a different skill, change `REVIEW_GUARD`, the data and `grade`. The measurement and the solve stay
the same. The method and the theory are in [the paper](https://arxiv.org/abs/2610.29419).